In [ ]:
# --- NEW CELL 1 (Installations ONLY) ---

# 1. UNINSTALL all conflicting packages for a clean slate
!pip uninstall -y opencv-python opencv-python-contrib numpy scipy scikit-learn ultralytics

# 2. RUN ONE "MASTER" COMMAND
!pip install ultralytics scikit-learn "numpy<2.0" opencv-python

print("--- Installation Cell Complete ---")

In [ ]:
# --- NEW CELL 2 (Imports and Parameters) ---

import os
import cv2
import glob
import json
import math
import torch
import torchvision.transforms as T
import torchvision.models as models
import numpy as np
from PIL import Image
from tqdm import tqdm
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from ultralytics import YOLO

# --- Parameters for your "Specialist" Model ---
# EXTRACTION_FPS is no longer needed for the "all-in-memory" function
# We run the "Spotter" on every Nth frame, defined inside process_sample
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

YOLO_CONF_THRESHOLD = 0.25 
# (Was 0.55) Lower this to be a "sanity check" (pro-tip from last time)
EMBED_SIM_THRESHOLD = 0.30          

MIN_TRACK_LENGTH_FRAMES = 5
MAX_KEYPOINTS_TO_TRACK = 100
# We will hard-code the Spotter FPS inside the function for clarity
SPOTTER_FPS = 20 # This is the 20 FPS from your 0.1818 score

print('Device:', DEVICE)
print("--- Import Cell Complete ---")

In [ ]:
# --- NEW CELL 3 (Data Preparation for YOLOv8) ---

import shutil

# --- 1. Define Paths ---
DATASET_ROOT = '/kaggle/input/zaic2025-observing/train'
ANNOTATIONS_FILE = '/kaggle/input/zaic2025-observing/train/annotations/annotations.json'
SAMPLES_DIR = os.path.join(DATASET_ROOT, 'samples')

YOLO_DATASET_DIR = '/kaggle/working/yolo_dataset'

if os.path.exists(YOLO_DATASET_DIR):
    shutil.rmtree(YOLO_DATASET_DIR)

os.makedirs(os.path.join(YOLO_DATASET_DIR, 'images', 'train'), exist_ok=True)
os.makedirs(os.path.join(YOLO_DATASET_DIR, 'images', 'val'), exist_ok=True)
os.makedirs(os.path.join(YOLO_DATASET_DIR, 'labels', 'train'), exist_ok=True)
os.makedirs(os.path.join(YOLO_DATASET_DIR, 'labels', 'val'), exist_ok=True)

# --- 2. Create Class-to-ID Mapping ---
all_sample_dirs = [d for d in os.listdir(SAMPLES_DIR) if os.path.isdir(os.path.join(SAMPLES_DIR,d))]
class_names = sorted(list(set([d.split('_')[0] for d in all_sample_dirs])))
class_map = {name: i for i, name in enumerate(class_names)}

print("Found class names:", class_map)

# --- 3. BBox Conversion Function (Unchanged) ---
def convert_to_yolo_format(x1, y1, x2, y2, img_w, img_h):
    box_w = x2 - x1
    box_h = y2 - y1
    x_center = x1 + (box_w / 2)
    y_center = y1 + (box_h / 2)
    
    x_c_norm = x_center / img_w
    y_c_norm = y_center / img_h
    w_norm = box_w / img_w
    h_norm = box_h / img_h
    
    return x_c_norm, y_c_norm, w_norm, h_norm

# --- 4. Main Processing Loop (CORRECTED) ---
print("Loading annotations...")
with open(ANNOTATIONS_FILE, 'r') as f:
    all_annotations = json.load(f) # This is a LIST

print("Starting data conversion...")
# --- THIS IS THE FIX: We iterate over the LIST directly ---
pbar = tqdm(all_annotations, desc="Processing videos")
for video_data in pbar:
    video_id = video_data['video_id']
    annotations_list = video_data['annotations'] # This is a list, e.g., [{"bboxes": [...]}]
    
    pbar.set_description(f"Processing {video_id}")
    
    video_path = os.path.join(SAMPLES_DIR, video_id, 'drone_video.mp4')
    if not os.path.exists(video_path):
        print(f"Warning: Video not found for {video_id}, skipping.")
        continue
        
    class_name = video_id.split('_')[0]
    class_id = class_map[class_name]
    
    split = 'val' if video_id.endswith('_1') else 'train'
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Warning: Cannot open video {video_id}, skipping.")
        continue
    
    img_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    img_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # --- THIS IS THE 2ND FIX: Iterate through the nested structure ---
    frames_to_extract = {}
    for track in annotations_list: # This is the {"bboxes": [...]} object
        for ann in track['bboxes']: # This is the {"frame": ...} object
            frame_idx = ann['frame']
            if frame_idx not in frames_to_extract:
                frames_to_extract[frame_idx] = []
            
            # Add the class_id to the annotation for the label file
            ann_with_class = ann.copy()
            ann_with_class['class_id'] = class_id
            frames_to_extract[frame_idx].append(ann_with_class)

    # Extract *only* the frames we need
    for frame_idx, anns_for_frame in frames_to_extract.items():
        img_name = f"{video_id}_frame_{frame_idx:06d}.jpg"
        label_name = f"{video_id}_frame_{frame_idx:06d}.txt"
        
        img_save_path = os.path.join(YOLO_DATASET_DIR, 'images', split, img_name)
        label_save_path = os.path.join(YOLO_DATASET_DIR, 'labels', split, label_name)
        
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if ret:
            cv2.imwrite(img_save_path, frame)
            
            with open(label_save_path, 'w') as f:
                for ann in anns_for_frame:
                    x1, y1, x2, y2 = ann['x1'], ann['y1'], ann['x2'], ann['y2']
                    cid = ann['class_id']
                    yolo_coords = convert_to_yolo_format(x1, y1, x2, y2, img_w, img_h)
                    f.write(f"{cid} {' '.join(map(str, yolo_coords))}\n")
        else:
            print(f"Warning: Could not read frame {frame_idx} from {video_id}")
            
    cap.release()

print("--- Data Preparation Complete ---")
print(f"Dataset created at: {YOLO_DATASET_DIR}")

In [ ]:
# --- NEW CELL 4 (Create data.yaml) ---

# We need to pass the class_map from the previous cell to this one.
# Make sure to run the cell above first!
yaml_content = f"""
train: {YOLO_DATASET_DIR}/images/train
val: {YOLO_DATASET_DIR}/images/val

# number of classes
nc: {len(class_map)}

# class names
names: {list(class_map.keys())}
"""

with open('data.yaml', 'w') as f:
    f.write(yaml_content)

print("--- data.yaml created ---")
%cat data.yaml

In [ ]:
# --- NEW CELL 5 (Run Training with Heavy Augmentation) ---

yolo_model = YOLO('yolov8n.pt') # Load the pre-trained nano model

print("--- Starting YOLOv8 Fine-Tuning (with Heavy Augmentation) ---")

# This will train for a *maximum* of 50 epochs,
# but will stop early if the validation score doesn't improve for 5 epochs.
yolo_model.train(
    data='data.yaml',
    epochs=10,
    patience=5,  # --- NEW: Enable Early Stopping ---
    imgsz=640,
    batch=16, 
    device=DEVICE,
    
    # --- NEW: Add Heavy Augmentation ---
    degrees=30,     # random rotation (+/- 30 degrees)
    translate=0.1,  # random translation (+/- 10%)
    scale=0.4,      # random scale (+/- 40%)
    flipud=0.5,     # random vertical flip (50% chance)
    hsv_h=0.015,    # change hue
    hsv_s=0.7,      # change saturation
    hsv_v=0.4       # change brightness
)
print("--- Training Complete ---")

In [ ]:
# --- NEW CELL 6 (Build ResNet50 embedding model) ---

# Build ResNet50 embedding model (global avgpool features)
import torch.nn as nn

def build_resnet50_embedding(device=DEVICE):
    model = models.resnet50(pretrained=True)
    # remove classifier, keep up to avgpool
    modules = list(model.children())[:-1]  # remove fc
    feat_extractor = nn.Sequential(*modules)
    feat_extractor.eval()
    feat_extractor.to(device)
    return feat_extractor

transform = T.Compose([
    T.ToPILImage(),
    T.Resize((224,224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

def image_to_embedding(img_bgr, model, device=DEVICE):
    # img_bgr: numpy BGR image (cv2)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    inp = transform(img_rgb).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = model(inp)  # shape [1, 2048, 1, 1]
    feat = feat.squeeze().cpu().numpy()
    feat = feat.reshape(-1)
    # L2 normalize
    norm = np.linalg.norm(feat)
    if norm>0:
        feat = feat / norm
    return feat

In [ ]:
# --- NEW CELL 7 (compute_support_embeddings) ---

def compute_support_embeddings(support_paths, model):
    embeds = []
    for p in support_paths:
        img = cv2.imread(p)
        if img is None:
            raise RuntimeError('Cannot read support image: ' + p)
        e = image_to_embedding(img, model)
        embeds.append(e)
    if len(embeds)==0:
        return None
    # average prototype
    proto = np.mean(np.stack(embeds), axis=0)
    proto = proto / (np.linalg.norm(proto)+1e-9)
    return proto, embeds

In [ ]:
# --- NEW CELL 8 (All-in-Memory, Original 0.1818 Chaser Logic) ---

def process_sample(sample_dir, feat_model, yolo_model):
    video_path = os.path.join(sample_dir, 'drone_video.mp4')
    support_glob = os.path.join(sample_dir, 'object_images', '*.jpg')
    support_paths = sorted(glob.glob(support_glob))
    if len(support_paths)==0:
        return {'video_id': os.path.basename(sample_dir), 'detections': []}
    video_id = os.path.basename(sample_dir)
        
    proto, _ = compute_support_embeddings(support_paths, feat_model)
    all_detections = []
    
    # --- 1. OPEN VIDEO ONCE ---
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Cannot open video {video_id}")
        return {'video_id': video_id, 'detections': []}
        
    video_fps = cap.get(cv2.CAP_PROP_FPS) or 30
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    # Get the "Spotter" step rate from our parameter
    spotter_step = max(1, int(round(video_fps / SPOTTER_FPS)))
    
    # Parameters for Lucas-Kanade Optical Flow "Chaser"
    lk_params = dict( winSize  = (15, 15),
                      maxLevel = 2,
                      criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))
    
    current_frame_id = 0
    pbar = tqdm(total=total_frames, desc='Spotting')
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break # End of video
            
        pbar.update(1)
        
        # --- 2. "SPOTTER" LOGIC ---
        if current_frame_id % spotter_step == 0:
            
            candidates = []
            yolo_results = yolo_model(frame, conf=YOLO_CONF_THRESHOLD, verbose=False)
            for box in yolo_results[0].boxes:
                xyxy = box.xyxy[0].cpu().numpy().astype(int)
                x1, y1, x2, y2 = xyxy
                crop = frame[y1:y2, x1:x2]
                if crop.size == 0: continue
                emb = image_to_embedding(crop, feat_model)
                sim = float(np.dot(proto, emb))
                candidates.append({'sim': sim, 'box': [x1, y1, x2, y2]})
            
            if len(candidates) > 0:
                candidates = sorted(candidates, key=lambda x: x['sim'], reverse=True)
                best = candidates[0]
                
                # --- 3. "CHASER" ACTIVATION (Original "min/max" logic) ---
                if best['sim'] >= EMBED_SIM_THRESHOLD:
                    print(f"\n  > Spotter found object at frame {current_frame_id}. Activating Chaser (min/max).")
                    
                    box = best['box']
                    x1, y1, x2, y2 = box
                    
                    # This is the line that had the SyntaxError
                    current_track = [{'frame': int(current_frame_id), 'x1': int(x1), 'y1': int(y1), 'x2': int(x2), 'y2': int(y2)}]

                    old_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                    mask = np.zeros_like(old_gray)
                    mask[y1:y2, x1:x2] = 255
                    p0 = cv2.goodFeaturesToTrack(old_gray, mask=mask, maxCorners=MAX_KEYPOINTS_TO_TRACK, qualityLevel=0.3, minDistance=7, blockSize=7)

                    if p0 is None:
                        print(f"  > Chaser failed: No keypoints found in initial box.")
                        current_frame_id += 1
                        continue # Go back to spotting

                    # --- 4. "CHASER" LOOP ---
                    while True:
                        ret_chase, chaser_frame = cap.read()
                        if not ret_chase:
                            break # End of video
                            
                        current_frame_id += 1 
                        pbar.update(1)
                        
                        frame_gray = cv2.cvtColor(chaser_frame, cv2.COLOR_BGR2GRAY)
                        p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)
                        
                        if p1 is not None:
                            good_new = p1[st==1]
                        
                        if p1 is None or len(good_new) < 4:
                            print(f"  > Chaser failed: Lost keypoints.")
                            break
                        
                        old_gray = frame_gray.copy()
                        p0 = good_new.reshape(-1, 1, 2)
                        
                        # --- Your 0.1818 "Shaky" Logic ---
                        x_min, y_min = np.min(good_new, axis=0)
                        x_max, y_max = np.max(good_new, axis=0)
                        
                        current_track.append({
                            'frame': int(current_frame_id),
                            'x1': int(x_min), 'y1': int(y_min),
                            'x2': int(x_max), 'y2': int(y_max)
                        })
                    
                    # --- 5. TRACK POST-PROCESSING ---
                    if len(current_track) >= MIN_TRACK_LENGTH_FRAMES:
                        all_detections.append({'bboxes': current_track})
                        print(f"  > Chaser finished. Logged track of {len(current_track)} frames.")

        current_frame_id += 1 
    
    pbar.close()
    cap.release()
            
    return {'video_id': video_id, 'detections': all_detections}

In [ ]:
# --- NEW CELL 9 (MAIN - All Fixes Included) ---

# MAIN: run on all samples under a dataset root, save submission.json
DATASET_ROOT = '/kaggle/input/zaic2025-publictest/public_test'

# --- THIS IS THE FIX FOR THE "NameError" ---
samples_root = os.path.join(DATASET_ROOT, 'samples')
sample_dirs = sorted([os.path.join(samples_root, d) for d in os.listdir(samples_root) if os.path.isdir(os.path.join(samples_root,d))])
print('Found', len(sample_dirs), 'samples')
# --- END FIX ---

print('Building ResNet50 feature extractor...')
feat_model = build_resnet50_embedding(DEVICE)

# --- THIS IS THE FIX TO USE YOUR NEW MODEL ---
TRAINED_MODEL_PATH = '/kaggle/working/runs/detect/train/weights/best.pt'
print(f'Loading custom-trained YOLO model from {TRAINED_MODEL_PATH}...')

if os.path.exists(TRAINED_MODEL_PATH):
    yolo_model = YOLO(TRAINED_MODEL_PATH) 
else:
    print("CRITICAL Warning: Custom model not found. Falling back to yolov8n.pt.")
    yolo_model = YOLO('yolov8n.pt')
# --- END FIX ---

yolo_model.to(DEVICE)
print('YOLO model loaded.')

results = []

for sd in sample_dirs:
    print('\nProcessing sample:', sd)
    # This call is now correct (no "OUT_BASE")
    res = process_sample(sd, feat_model, yolo_model)
    results.append(res)

# write submission
submission_path = 'submission.json' # Save to root
with open(submission_path, 'w') as f:
    json.dump(results, f, indent=2)
print('Saved submission to', submission_path)